In [2]:
import pandas as pd
import os
from langchain.chat_models import init_chat_model
from time import time
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
labeled_data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/data/datasets/labeled_data.csv')
labeled_data = labeled_data[labeled_data['verified_pattern'].notna()]
labeled_data['path'] = labeled_data['file'].apply(lambda x: f'/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/repo_callgraph_clusters/{x.split("/")[-2]}/{x.split("/")[-1]}')

def load_code(path):
    with open(path, 'r') as file:
        return file.read()
labeled_data['code'] = labeled_data['path'].apply(load_code)

In [4]:
gem = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0.7)
gpt = init_chat_model("gpt-5-nano", model_provider="openai", temperature=0.7)

In [5]:
prompt = """\
You are an expert code description generator. Generate a concise and informative description of the community in 2-3 sentences. Focus on the code AI patterns. Strictly generate the description about patterns represented in the code. Avoid generic statements and ensure the description is specific to a AI patterns.

code:
{code}
"""

In [ ]:
def generate_community_description(code: str,llm,max_retries=3) -> str:
    for _ in range(max_retries):
        try:
            print(f"--------- Generating code summary",end="\r")
            response = llm.invoke(prompt.format(code=code))
            print(f"Generated --------------",end="\r")
            return response.content.strip()
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(1)

from concurrent.futures import ThreadPoolExecutor

def generate_code_summaries(code: str) -> str:
    def task(llm):
        return generate_community_description(code, llm)

    print("Generating code summaries (6 in parallel)...", end="\r")
    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = [
            executor.submit(task, gem),  # 01
            executor.submit(task, gpt),  # 02
            executor.submit(task, gem),  # 03
            executor.submit(task, gpt),  # 04
            executor.submit(task, gem),  # 05
            executor.submit(task, gpt),  # 06
        ]
        results = [f.result() for f in futures]
    return tuple(results)

In [7]:
from torch._dynamo.pgo import code_state_path


description_file = "result/community_description/feb-10-2026-community-descriptions.csv"
if os.path.exists(description_file):
    descriptions = pd.read_csv(description_file)
else:
    descriptions = pd.DataFrame(columns=['file', 'code','verified_pattern','code_summary_01','code_summary_02','code_summary_03','code_summary_04','code_summary_05','code_summary_06'])


def main(file,code,verified_pattern):
    print(f"{len(descriptions)} - Generating code summary for {file}")
    if file in descriptions['file'].values:
        return 0
    description = generate_code_summaries(code)
    new_dict = {}
    new_dict['code_summary_01'] = description[0]
    new_dict['code_summary_02'] = description[1]
    new_dict['code_summary_03'] = description[2]
    new_dict['code_summary_04'] = description[3]
    new_dict['code_summary_05'] = description[4]
    new_dict['code_summary_06'] = description[5]
    new_dict['verified_pattern'] = verified_pattern
    new_dict['file'] = file
    new_dict['code'] = code
    descriptions.loc[len(descriptions)] = new_dict
    descriptions.to_csv(description_file,index=False)

In [ ]:
for index,row in labeled_data.iterrows():
    main(row['file'],row['code'],row['verified_pattern'])


17 - Generating code summary for https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/3DOD_thesis/cluster_0.py
17 - Generating code summary for https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/3DOD_thesis/cluster_1.py
17 - Generating code summary for https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/3DOD_thesis/cluster_10.py
17 - Generating code summary for https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/3DOD_thesis/cluster_3.py
17 - Generating code summary for https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_clusters/3DOD_thesis/cluster_4.py
17 - Generating code summary for https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/tree/main/notebooks/result/repo_callgraph_c

In [13]:
labeled_data

,Unnamed: 0,verified_pattern,code_summary,verified,file,path,code
0,0,LLM-based Multimodal Generative Prompting,This code implements a data pipeline for 3D ob...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,"# Cluster 0\n\ndef getBinNumber(angle, NH):\n ..."
1,1,none,This code implements a 3D object pose estimati...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 1\n\nclass BoxRegressor(object):\n\n...
2,2,LLM-based Multimodal Generative Prompting,This code defines data pipelines for 3D object...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 10\n\ndef LabelLoader2D3D_sequence(i...
4,4,Preprocessing Text and Numerical Data,This code implements an **angular discretizati...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 3\n\ndef getBinNumber4(angle):\n ...
5,5,none,This code defines a PyTorch Dataset for 3D obj...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 4\n\nclass DatasetFrustumPointNetAug...
...,...,...,...,...,...,...,...
1662,1662,Retrieval Augmented Generation(RAG),This code implements a modular AI processing s...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 10\n\nclass Vigil:\n vectordb: Op...
1665,1665,Retrieval Augmented Generation(RAG),"This code establishes a modular AI pipeline, i...",True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 2\n\nclass Vigil:\n vectordb: Opt...
1667,1667,Retrieval Augmented Generation(RAG),This code implements a vector database pattern...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 7\n\nclass VectorDB:\n\n def __in...
1668,1668,LLM-based User Intent Extraction,The `Vigil` class implements an **AI orchestra...,True,https://github.com/HasinthakaPiyumal/AI-Patter...,/home/hasinthaka/Documents/Projects/AI/Pattern...,# Cluster 9\n\nclass Vigil:\n vectordb: Opt...


In [3]:
import os,pandas as pd
description_file = "result/community_description/feb-10-2026-community-descriptions.csv"
if os.path.exists(description_file):
    descriptions = pd.read_csv(description_file)

In [7]:
concated_descriptions = pd.DataFrame(columns=['file','verified_pattern','code_summary'])

In [8]:
for i in range(6):
    temp_descriptions = pd.DataFrame(columns=['file','verified_pattern','code_summary'])
    temp_descriptions['file'] = descriptions['file']
    temp_descriptions['verified_pattern'] = descriptions['verified_pattern']
    temp_descriptions['code_summary'] = descriptions[f'code_summary_0{i+1}']
    concated_descriptions = pd.concat([concated_descriptions,temp_descriptions])
concated_descriptions.to_csv("result/community_description/feb-10-2026-community-descriptions-concated.csv",index=False)


In [9]:
concated_descriptions

,file,verified_pattern,code_summary
0,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM-based Multimodal Generative Prompting,This code implements a data preparation pipeli...
1,https://github.com/HasinthakaPiyumal/AI-Patter...,none,This code demonstrates a model-based 3D object...
2,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM-based Multimodal Generative Prompting,This code implements data preprocessing patter...
3,https://github.com/HasinthakaPiyumal/AI-Patter...,Preprocessing Text and Numerical Data,This code implements an **orientation binning ...
4,https://github.com/HasinthakaPiyumal/AI-Patter...,none,This code implements a PyTorch dataset specifi...
...,...,...,...
408,https://github.com/HasinthakaPiyumal/AI-Patter...,Retrieval Augmented Generation(RAG),"Vigil showcases a config-driven, plug-in AI wo..."
409,https://github.com/HasinthakaPiyumal/AI-Patter...,Retrieval Augmented Generation(RAG),"Vigil demonstrates a config-driven, plugin-bas..."
410,https://github.com/HasinthakaPiyumal/AI-Patter...,Retrieval Augmented Generation(RAG),The code showcases a pluggable embedding strat...
411,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM-based User Intent Extraction,The code demonstrates a registry-driven plugin...
